# Pooled cohorts: correlations and feature selection (Kenya + Tanzania)

This notebook settles the open questions from the country EDAs and proposes the `SELECTED` column list for each pooled cohort before modelling.

- **Data:** the three pooled cohorts from `dataset.build_cohort()` (Kenya + Tanzania stacked, cleaned, with the agreed 80/20 cluster split).
- **Training rows only.** Every table, correlation and chart below uses `split == "train"`. The test clusters are not looked at, so nothing here can leak into the final evaluation.
- **Rates** are survey-weighted (`weight`), per 1,000, with an approximate 95% CI (Wilson interval on the Kish effective sample size). Groups with fewer than 30 events are flagged (`*` in the compact tables, grey in the charts): read them with caution.
- **Outcome:** `perinatal_death` for the perinatal and perinatal ANC cohorts; `low_birth_weight` for the LBW cohort.
- Only aggregates are shown: counts, weighted rates and correlations. No individual rows.

Sections:
- **A.** Open question 1: `anc_visits` or `anc4plus`? (is the visits gradient just gestation?)
- **B.** Open question 2: wealth and residence (real effect or reporting?)
- **C.** Open question 3: parity (U-shape, and how to encode it)
- **D.** Open question 4: early booking (reverse causation?)
- **E.** Other checks: very young mothers, birth-weight heaping, imputed intervals
- **F.** Correlations between candidate features, and each one's univariate signal
- **G.** Summary: proposed `SELECTED` lists

## 1. Set-up

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# The notebook lives in ml/notebooks/pooled/, so ml/src is two folders up.
sys.path.append(str(Path.cwd().parents[1] / "src"))
from dataset import build_cohort  # noqa: E402

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

## 2. Load the three pooled cohorts and keep the training rows

We check the full row and event counts first (they must match the agreed numbers), then keep the training rows only.

In [ ]:
COHORTS = ["perinatal", "perinatal_anc", "lbw"]
OUTCOMES = {
    "perinatal": "perinatal_death",
    "perinatal_anc": "perinatal_death",
    "lbw": "low_birth_weight",
}
EXPECTED = {"perinatal": (30802, 1089), "perinatal_anc": (16542, 520), "lbw": (8925, 711)}

train = {}
summary = []
for c in COHORTS:
    df = build_cohort(c)
    rows, events = len(df), int(df[OUTCOMES[c]].sum())
    assert (rows, events) == EXPECTED[c], f"{c}: got {rows}/{events}, expected {EXPECTED[c]}"
    tr = df[df["split"] == "train"].reset_index(drop=True)
    train[c] = tr
    summary.append(
        {
            "cohort": c,
            "rows (all)": rows,
            "events (all)": events,
            "rows (train)": len(tr),
            "events (train)": int(tr[OUTCOMES[c]].sum()),
            "Kenya train rows": int((tr["country"] == "Kenya").sum()),
            "Tanzania train rows": int((tr["country"] == "Tanzania").sum()),
        }
    )
pd.DataFrame(summary).set_index("cohort")

### Which columns are candidates?

**Never modelled** (never candidate features): `region`, `told_high_bp`, `prior_pregnancies`, `country`, `pregnancy_months`, `birth_weight_g`, `delivery_place`, `baby_sex`, `stillbirth`, `early_neonatal_death`, `pregnancy_outcome`, the raw DHS code columns (`v106`, `m14` and so on), `caseid`, `pidx`, `v001`. `country` is still kept in `SELECTED` as a design column (for stratifying and reporting), not as a feature. `pregnancy_months` and `delivery_place` appear below **only as stratifiers in checks**, and are labelled as such.

The cell below lists which candidate features actually exist in each cohort.

In [ ]:
BASE_CANDIDATES = [
    "maternal_age",
    "prior_live_births",
    "prior_stillbirth",
    "prior_loss",
    "preceding_interval_m",
    "first_pregnancy",
    "interval_imputed",
    "interval_group",
    "short_interval",
    "multiple",
    "education",
    "wealth_quintile",
    "residence",
]
ANC_CANDIDATES = [
    "anc_visits",
    "no_anc",
    "anc4plus",
    "first_anc_month",
    "bp_measured",
    "urine_tested",
    "blood_tested",
    "iron_tablets",
]
candidates = {}
for c in COHORTS:
    wanted = BASE_CANDIDATES + (ANC_CANDIDATES if c != "perinatal" else [])
    candidates[c] = [v for v in wanted if v in train[c].columns]
    missing = [v for v in wanted if v not in train[c].columns]
    print(f"{c}: {len(candidates[c])} candidates present; missing: {missing or 'none'}")
    print("   ", ", ".join(candidates[c]))

### Helpers and analysis groupings

The rate helpers are the same as in the country EDAs (`weighted_rate`, `rate_ci`, `rate_by`), with the outcome passed in so they work for both deaths and low birth weight. `rate_table` adds pooled and by-country (or any other) strata, and `compact` turns that into one readable cell per group: `rate [CI] (events)`, with `*` when there are fewer than 30 events.

The groupings (`age_group`, `parity_group`, `anc_group`, `first_anc_trimester`, `delivery_type`) are for these tables only; they are not added to the cohort files.

In [ ]:
def weighted_rate(d, outcome, per=1000):
    """Survey-weighted rate per 1,000 (or per 100 with per=100)."""
    return per * np.average(d[outcome], weights=d["weight"])


def rate_ci(d, outcome, per=1000):
    """Weighted rate with an approximate 95% CI (Wilson, Kish effective sample size)."""
    w, y = d["weight"], d[outcome]
    p = np.average(y, weights=w)
    n_eff = w.sum() ** 2 / (w**2).sum()
    z = 1.96
    centre = (p + z**2 / (2 * n_eff)) / (1 + z**2 / n_eff)
    half = z * np.sqrt(p * (1 - p) / n_eff + z**2 / (4 * n_eff**2)) / (1 + z**2 / n_eff)
    return per * p, per * max(centre - half, 0), per * (centre + half)


def rate_by(d, var, outcome, order=None):
    """Weighted outcome rate per 1,000 within each level of `var`."""
    rows = []
    for level, g in d.dropna(subset=[var]).groupby(var, observed=True):
        rate, lo, hi = rate_ci(g, outcome)
        events = int(g[outcome].sum())
        rows.append(
            {
                "level": level,
                "n": len(g),
                "events": events,
                "rate_per_1000": rate,
                "ci_low": lo,
                "ci_high": hi,
                "few_events": events < 30,
            }
        )
    out = pd.DataFrame(rows).set_index("level")
    return out.reindex(order) if order is not None else out


def rate_table(d, var, outcome, order=None, by="country"):
    """rate_by for all rows ("Pooled") and within each level of `by`."""
    parts = {"Pooled": rate_by(d, var, outcome, order)}
    if by is not None:
        for level, g in d.dropna(subset=[by]).groupby(by, observed=True):
            parts[str(level)] = rate_by(g, var, outcome, order)
    return pd.concat(parts, names=["stratum", var])


def compact(t):
    """One cell per group and stratum: 'rate [low-high] (events)', * if < 30 events."""
    def fmt(r):
        if pd.isna(r["rate_per_1000"]):
            return ""
        flag = "*" if r["few_events"] else ""
        return (
            f"{r['rate_per_1000']:.0f} [{r['ci_low']:.0f}-{r['ci_high']:.0f}] "
            f"({int(r['events'])}){flag}"
        )
    s = t.apply(fmt, axis=1)
    return s.unstack(0).reindex(columns=t.index.get_level_values(0).unique())


def plot_strata(t, title, ylabel):
    """Grouped bars: one group per level, one bar per stratum, with 95% CI."""
    strata = list(t.index.get_level_values(0).unique())
    levels = list(t.index.get_level_values(1).unique())
    x = np.arange(len(levels))
    width = 0.8 / len(strata)
    fig, ax = plt.subplots(figsize=(max(6, 1.4 * len(levels) * len(strata) / 2), 3.8))
    for i, s in enumerate(strata):
        sub = t.loc[s].reindex(levels)
        err = [sub["rate_per_1000"] - sub["ci_low"], sub["ci_high"] - sub["rate_per_1000"]]
        bars = ax.bar(x + i * width, sub["rate_per_1000"], width, yerr=err, capsize=2, label=s)
        for b, few in zip(bars, sub["few_events"].fillna(False)):
            if few:
                b.set_alpha(0.35)
    ax.set_xticks(x + width * (len(strata) - 1) / 2, [str(v) for v in levels], rotation=20)
    ax.set_ylabel(ylabel)
    ax.set_title(title + "\n(faded = fewer than 30 events; bars = 95% CI)")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


def ylabel(c):
    return "Deaths per 1,000" if OUTCOMES[c] == "perinatal_death" else "LBW per 1,000"


for c, d in train.items():
    d["age_group"] = pd.cut(
        d["maternal_age"], [0, 17, 24, 34, 60], labels=["<18", "18-24", "25-34", "35+"]
    )
    d["parity_group"] = pd.cut(
        d["prior_live_births"], [-1, 0, 2, 4, 30], labels=["0", "1-2", "3-4", "5+"]
    )
    if "anc_visits" in d:
        d["anc_group"] = pd.cut(
            d["anc_visits"], [-1, 0, 3, 7, 40], labels=["None", "1-3", "4-7", "8+"]
        )
        d["first_anc_trimester"] = pd.cut(
            d["first_anc_month"],
            [-1, 0, 3, 6, 10],
            labels=["No ANC", "1st (0-3 m)", "2nd (4-6 m)", "3rd (7+ m)"],
        )
    if "delivery_place" in d:
        # Check-only stratifier: facility (public, private, NGO/faith) vs home; "Other" left out.
        d["delivery_type"] = d["delivery_place"].map(
            {
                "Public facility": "Facility",
                "Private facility": "Facility",
                "NGO or faith-based facility": "Facility",
                "Home": "Home",
            }
        )

ORDERS = {
    "wealth_quintile": ["Poorest", "Poorer", "Middle", "Richer", "Richest"],
    "residence": ["Urban", "Rural"],
    "education": ["No education", "Primary", "Secondary", "Higher"],
    "parity_group": ["0", "1-2", "3-4", "5+"],
    "first_pregnancy": [0, 1],
    "anc_group": ["None", "1-3", "4-7", "8+"],
    "first_anc_trimester": ["No ANC", "1st (0-3 m)", "2nd (4-6 m)", "3rd (7+ m)"],
}

pd.DataFrame(
    {c: {"overall rate per 1,000": weighted_rate(d, OUTCOMES[c])} for c, d in train.items()}
).T

## A. Open question 1: `anc_visits` or `anc4plus`?

More visits looked protective in the country EDAs. But a woman who delivers at 7 months has had less time to attend, so a short pregnancy (itself a strong risk for death and low birth weight) automatically means fewer visits. If that is the whole story, the visits "effect" is partly gestation in disguise.

`pregnancy_months` is used here **only as a stratifier for this check** (it is measured at the end of the pregnancy and is never a feature). Checks, for the perinatal ANC and LBW cohorts:
1. weighted mean visits and outcome rate by pregnancy length (7, 8, 9 months; 10 is shown for completeness);
2. outcome rate by `anc_group`, all pregnancies vs 9-month pregnancies only;
3. Spearman correlation of `anc_visits` with `pregnancy_months`.

In [ ]:
def by_months(d, outcome):
    rows = []
    for m, g in d.groupby("pregnancy_months"):
        known = g.dropna(subset=["anc_visits"])
        rate, lo, hi = rate_ci(g, outcome)
        rows.append(
            {
                "pregnancy_months (check only)": int(m),
                "n": len(g),
                "mean_visits_weighted": np.average(known["anc_visits"], weights=known["weight"]),
                "anc4plus_%": 100 * np.average(known["anc4plus"], weights=known["weight"]),
                "events": int(g[outcome].sum()),
                "rate_per_1000": rate,
                "ci_low": lo,
                "ci_high": hi,
                "few_events": int(g[outcome].sum()) < 30,
            }
        )
    return pd.DataFrame(rows).set_index("pregnancy_months (check only)")


for c in ["perinatal_anc", "lbw"]:
    print(f"\n=== {c} ({OUTCOMES[c]}) ===")
    display(by_months(train[c], OUTCOMES[c]))

In [ ]:
anc_9m = {}
for c in ["perinatal_anc", "lbw"]:
    d, y = train[c], OUTCOMES[c]
    t = pd.concat(
        {
            "All pregnancies": rate_by(d, "anc_group", y, ORDERS["anc_group"]),
            "9-month only": rate_by(d[d["pregnancy_months"] == 9], "anc_group", y, ORDERS["anc_group"]),
        },
        names=["stratum", "anc_group"],
    )
    anc_9m[c] = t
    rho = d[["anc_visits", "pregnancy_months"]].corr(method="spearman").iloc[0, 1]
    print(f"\n=== {c}: Spearman(anc_visits, pregnancy_months) = {rho:.3f} ===")
    display(compact(t))
    plot_strata(t, f"{c}: {y} by ANC visits, all vs 9-month pregnancies", ylabel(c))

In [ ]:
# The same comparison by country, 9-month pregnancies only.
for c in ["perinatal_anc", "lbw"]:
    d = train[c]
    t = rate_table(d[d["pregnancy_months"] == 9], "anc_group", OUTCOMES[c], ORDERS["anc_group"])
    print(f"\n=== {c}: 9-month pregnancies only, by country ===")
    display(compact(t))

**Result:** pregnancy length explains most of the visits gradient for perinatal death, but not all of it for low birth weight.

- **Shorter pregnancies have fewer visits:** weighted mean 3.3 (perinatal ANC) and 3.5 (LBW) visits at 7 months, against 4.2 and 4.3 at 9 months; 4+ visits 38% at 7 months vs 66 to 71% at 9 months. Their outcome rates are far higher (deaths 387 per 1,000 at 7 months vs 22 at 9; LBW 735 vs 68).
- **Spearman of `anc_visits` with `pregnancy_months`** is small (0.06 perinatal ANC, 0.08 LBW): most pregnancies are 9 months long, so the overall rank correlation is weak, but the short pregnancies are where the deaths are.
- **Perinatal ANC cohort, deaths per 1,000 by visits:** all pregnancies None 58 [41-83] (48 deaths), 1-3 36 [30-45] (141), 4-7 26 [21-31] (211), 8+ 22* (11). In **9-month pregnancies only** the gradient flattens: None 36* (27), 1-3 19 [14-26] (64), 4-7 22 [18-27] (155), 8+ 21* (8). Only the no-ANC excess remains, on fewer than 30 deaths. Same pattern in both countries.
- **LBW cohort, LBW per 1,000:** all pregnancies None 100 [68-144] (32), 1-3 100 [84-119] (185), 4-7 70 [60-80] (339), 8+ 119* (25). In 9-month pregnancies: None 67*, 1-3 84 [69-102] (143), 4-7 62 [53-72] (285), 8+ 79* (19). A 1-3 vs 4-7 gap survives (84 vs 62, CIs just touch), in both countries.

**Conclusion:** for perinatal death the visits gradient does **not** survive in 9-month pregnancies; it is mostly gestation (women who deliver early have less time to attend). A count of visits would partly encode the outcome's timing. For LBW a modest 4+ visit advantage does survive. A threshold (`anc4plus`, plus `no_anc`) captures what signal there is; `anc_visits` adds little beyond it (Spearman 0.82 to 0.85 with `anc4plus`, section F).

## B. Open question 2: wealth, residence and education

Both country EDAs found richer, urban and better-educated women with *higher* recorded perinatal mortality. The leading explanation is reporting: women who deliver in a facility are more likely to have a stillbirth or early death recognised and recalled. If so, the gradient should largely disappear **within** each delivery place.

First the rates pooled and by country for all three cohorts; then the same stratified by facility vs home delivery. `delivery_place` is used **only as a stratifier for this check** (it is decided at the delivery, not known at booking). The headline perinatal cohort has no delivery place, so the stratified check is for the ANC and LBW cohorts only.

In [ ]:
SES = ["wealth_quintile", "residence", "education"]
ses_tables = {}
for c in COHORTS:
    for var in SES:
        t = rate_table(train[c], var, OUTCOMES[c], ORDERS[var])
        ses_tables[(c, var)] = t
        print(f"\n=== {c}: {OUTCOMES[c]} by {var} (pooled and by country) ===")
        display(compact(t))

In [ ]:
for c in COHORTS:
    fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
    for ax, var in zip(axes, SES):
        t = ses_tables[(c, var)]
        strata = list(t.index.get_level_values(0).unique())
        levels = ORDERS[var]
        x = np.arange(len(levels))
        w = 0.8 / len(strata)
        for i, s in enumerate(strata):
            sub = t.loc[s].reindex(levels)
            err = [sub["rate_per_1000"] - sub["ci_low"], sub["ci_high"] - sub["rate_per_1000"]]
            ax.bar(x + i * w, sub["rate_per_1000"], w, yerr=err, capsize=2, label=s)
        ax.set_xticks(x + w, levels, rotation=25)
        ax.set_title(var)
        ax.set_ylabel(ylabel(c))
    axes[0].legend(fontsize=8)
    fig.suptitle(f"{c}: {OUTCOMES[c]} by wealth, residence and education (bars = 95% CI)")
    plt.tight_layout()
    plt.show()

In [ ]:
# Check only: the same gradients within facility and home deliveries.
for c in ["perinatal_anc", "lbw"]:
    d = train[c]
    share_home = 100 * np.average(d["delivery_type"].eq("Home"), weights=d["weight"])
    print(f"\n##### {c}: weighted share delivered at home = {share_home:.1f}% #####")
    for var in SES:
        t = rate_table(d, var, OUTCOMES[c], ORDERS[var], by="delivery_type")
        print(f"\n=== {c}: {OUTCOMES[c]} by {var}, within delivery place (check only) ===")
        display(compact(t))

In [ ]:
# Wealth within delivery place, also split by country (perinatal ANC cohort).
d = train["perinatal_anc"]
for country, g in d.groupby("country"):
    t = rate_table(g, "wealth_quintile", "perinatal_death", ORDERS["wealth_quintile"], by="delivery_type")
    print(f"\n=== perinatal_anc, {country}: deaths by wealth within delivery place (check only) ===")
    display(compact(t))

**Result:** the "richer and urban = higher perinatal mortality" pattern is real in the data, comes mainly from Tanzania, and does **not** disappear within facility deliveries.

- **Perinatal cohort, deaths per 1,000 by wealth (pooled):** Poorest 30 [26-35] (237 deaths), Poorer 27 (122), Middle 36 (161), Richer 37 (180), Richest 39 [31-49] (169). Kenya is flat (22 to 34); Tanzania rises steeply (Poorest 25, Richest 57 [44-74]). **Residence:** Urban 38 [31-45] vs Rural 32 [29-35]; Tanzania 50 vs 35, Kenya 32 vs 30. **Education:** flat pooled (35, 35, 34, 28 from none to higher).
- **Perinatal ANC cohort:** same shape, weaker (wealth 28, 23, 30, 34, 35; Urban 35 vs Rural 28; education flat 26 to 32).
- **LBW cohort:** no "richer = worse" gradient. Wealth 86, 97, 74, 91, 61 (Richest lowest, 61 [46-81]); residence Urban 77 vs Rural 83 pooled, but in opposite directions by country (Kenya rural higher, 93 vs 64; Tanzania urban higher, 90 vs 74); higher education lowest (58 [37-89]).
- **Within delivery place (check only).** Only 14% of the perinatal ANC cohort (weighted) and 4% of the LBW cohort delivered at home, so the home strata are small (most cells under 30 events). Within **facility** deliveries the perinatal gradient persists: wealth 29, 25, 31, 36, 34; Urban 35 vs Rural 29. In Tanzanian facilities: 32, 38, 27, 41, 46. Home deliveries look lower in the poor groups (Poorest 23, Rural 21), but on few deaths.

**Conclusion:** stratifying by facility vs home does **not** remove the gradient, so the simple "facility deliveries record more deaths" explanation is not supported by this check. Reporting may still play a part in other ways (better recognition of stillbirths in urban hospitals, referral of complicated pregnancies to them), which this data cannot separate. Wealth and residence therefore carry some signal for perinatal death but its meaning is unclear and it runs against clinical expectation; for LBW the direction is the expected one (richest lowest) but inconsistent by country. **Proposed, Jeff to decide** (section G).

## C. Open question 3: parity

Both countries showed a U-shape: first births and grand multiparas (5+ previous births) at higher risk than parity 1 to 4. A single straight-line term in `prior_live_births` cannot capture a U. The question is whether **`first_pregnancy` + `prior_live_births`** together are enough (the flag takes the left arm, the count the right arm), or whether we need parity groups.

Note the two are not identical: `first_pregnancy` means no previous pregnancy at all, while parity 0 means no previous *live birth* (a woman can have had a loss but no live birth).

In [ ]:
parity_tables = {}
for c in COHORTS:
    for var in ["parity_group", "first_pregnancy"]:
        t = rate_table(train[c], var, OUTCOMES[c], ORDERS[var])
        parity_tables[(c, var)] = t
        print(f"\n=== {c}: {OUTCOMES[c]} by {var} (pooled and by country) ===")
        display(compact(t))
    plot_strata(parity_tables[(c, "parity_group")], f"{c}: {OUTCOMES[c]} by parity", ylabel(c))

In [ ]:
# How often does parity 0 differ from first_pregnancy? (counts only)
pd.DataFrame(
    {
        c: {
            "parity 0 rows": int(d["prior_live_births"].eq(0).sum()),
            "of which first_pregnancy = 1": int((d["prior_live_births"].eq(0) & d["first_pregnancy"].eq(1)).sum()),
            "parity 0 but a previous loss": int((d["prior_live_births"].eq(0) & d["first_pregnancy"].eq(0)).sum()),
        }
        for c, d in train.items()
    }
)

To test whether `first_pregnancy` + `prior_live_births` captures the shape, we fit small survey-weighted logistic regressions on the training rows and compare their weighted log-loss (lower is better):
- **linear:** `prior_live_births` alone;
- **flag + linear:** `first_pregnancy` + `prior_live_births`;
- **groups:** parity-group indicators (0, 1-2, 3-4, 5+), the most flexible.

If "flag + linear" is about as good as "groups", the two existing columns are enough and no parity-group column is needed.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss


def weighted_logloss(X, y, w):
    m = LogisticRegression(C=1e6, max_iter=2000).fit(X, y, sample_weight=w)
    return log_loss(y, m.predict_proba(X)[:, 1], sample_weight=w)


rows = {}
for c, d in train.items():
    y, w = d[OUTCOMES[c]], d["weight"]
    groups = pd.get_dummies(d["parity_group"], drop_first=True).astype(float)
    base = log_loss(y, np.full(len(y), np.average(y, weights=w)), sample_weight=w)
    rows[c] = {
        "intercept only": base,
        "linear": weighted_logloss(d[["prior_live_births"]], y, w),
        "flag + linear": weighted_logloss(d[["first_pregnancy", "prior_live_births"]], y, w),
        "groups": weighted_logloss(groups, y, w),
        "flag + linear + 5+ indicator": weighted_logloss(
            d[["first_pregnancy", "prior_live_births"]].assign(
                grand=d["prior_live_births"].ge(5).astype(float)
            ),
            y,
            w,
        ),
    }
ll = pd.DataFrame(rows).T
pd.options.display.float_format = "{:,.5f}".format
display(ll)
pd.options.display.float_format = "{:,.2f}".format

**Result:** the U-shape holds pooled for perinatal death; for LBW it is mainly a first-birth excess.

- **Perinatal cohort, deaths per 1,000 by previous live births:** 0: 38 [32-45] (243 deaths), 1-2: 29 [25-34] (280), 3-4: 34 [28-41] (174), 5+: 40 [33-49] (172). The left arm comes from Tanzania (0: 53 vs Kenya 30); the right arm from both (5+: 40 and 41).
- **Perinatal ANC cohort:** 0: 32 (114), 1-2: 28 (139), 3-4: 24 (75), 5+: 43 [33-56] (84). Again Tanzania gives the left arm (47 vs Kenya 26).
- **LBW cohort, LBW per 1,000:** 0: 92 [77-111] (172), 1-2: 78, 3-4: 70, 5+: 80. Mostly a first-birth excess.
- **`first_pregnancy` is not the same as parity 0:** 6 to 8% of parity-0 rows had a previous loss (424 / 233 / 144 rows). `first_pregnancy` rates: perinatal 36 vs 33; perinatal ANC 28 vs 31 (lower than parity 0); LBW 90 vs 77.
- **Weighted log-loss (training rows, in-sample):** a parity-group encoding is best for perinatal and perinatal ANC; `first_pregnancy` + linear `prior_live_births` gains almost nothing over linear alone (perinatal ANC 0.13573 for both); adding a 5+ indicator closes most of the gap (perinatal ANC 0.13546 vs groups 0.13539; LBW 0.27926, better than groups; less so in the headline cohort, 0.14756 vs 0.14742). Differences are small and in-sample.

**Conclusion:** the U holds pooled for perinatal death. `first_pregnancy` + `prior_live_births` does **not** fully capture it in a linear model: the right arm is a jump at 5+ rather than a straight-line rise. Tree models (random forest, XGBoost) can find this from `prior_live_births` on their own; for logistic regression the pipeline should bin `prior_live_births` (0, 1-2, 3-4, 5+) or add a 5+ indicator. No new column is needed in the cohort files.

## D. Open question 4: early booking

Women who booked ANC in the first trimester had a *higher* perinatal death rate in the country EDAs. The suspected cause is reverse causation: women who already know of a problem (above all a previous loss) book early. If so, the early-booking excess should shrink among women with **no** previous loss or stillbirth.

In [ ]:
booking = {}
for c in ["perinatal_anc", "lbw"]:
    d, y = train[c], OUTCOMES[c]
    order = ORDERS["first_anc_trimester"]
    parts = {"All": rate_by(d, "first_anc_trimester", y, order)}
    for v in ["prior_loss", "prior_stillbirth"]:
        for lv in [0, 1]:
            parts[f"{v}={lv}"] = rate_by(d[d[v] == lv], "first_anc_trimester", y, order)
    t = pd.concat(parts, names=["stratum", "first_anc_trimester"])
    booking[c] = t
    print(f"\n=== {c}: {y} by first_anc_trimester, overall and by previous loss ===")
    display(compact(t))
    plot_strata(
        t.loc[["All", "prior_loss=0", "prior_loss=1"]],
        f"{c}: {y} by trimester of first ANC visit",
        ylabel(c),
    )

In [ ]:
# Is a previous loss more common among early bookers? (weighted %)
for c in ["perinatal_anc", "lbw"]:
    d = train[c].dropna(subset=["first_anc_trimester"])
    out = d.groupby("first_anc_trimester", observed=True).apply(
        lambda g: pd.Series(
            {
                "n": len(g),
                "prior_loss_%": 100 * np.average(g["prior_loss"], weights=g["weight"]),
                "prior_stillbirth_%": 100 * np.average(g["prior_stillbirth"], weights=g["weight"]),
            }
        )
    )
    print(f"\n=== {c} ===")
    display(out)

In [ ]:
# Early booking by country (perinatal ANC cohort), women with no previous loss.
d = train["perinatal_anc"]
t = rate_table(d[d["prior_loss"] == 0], "first_anc_trimester", "perinatal_death", ORDERS["first_anc_trimester"])
compact(t)

**Result:** yes: the early-booking excess is explained by previous losses.

- **Perinatal ANC cohort, deaths per 1,000 by first ANC visit:** all women No ANC 58 [41-83] (48 deaths), 1st trimester 33 [26-42] (135), 2nd 28 [23-33] (211), 3rd 17* (17). Pooled, the 1st-trimester excess is smaller than in the Tanzania EDA (45 vs 28 there).
- **No previous loss (`prior_loss = 0`):** 1st 28 [21-37] (107) vs 2nd 27 [22-33] (174): the excess **disappears**. Same in each country (Kenya 25 vs 26; Tanzania 33 vs 29).
- **Previous loss (`prior_loss = 1`):** 1st 62* (28) vs 2nd 35 (37); previous stillbirth: 152* vs 86*. Early bookers more often have a previous loss (15% vs 10% of 2nd-trimester bookers, both cohorts).
- **LBW cohort:** no early-booking excess at all (1st 77 [64-93], 2nd 82 [71-94]); with no previous loss 70 vs 81.
- **3rd-trimester booking looks protective** (17* deaths per 1,000) because a woman can only book at 7+ months if the pregnancy has lasted that long: a built-in survival bias, not a benefit of late booking.

**Conclusion:** the early-booking excess shrinks to nothing among women with no previous loss, consistent with reverse causation. With `prior_loss` and `prior_stillbirth` in the model, `first_anc_month` adds no real signal, and its late values carry survival bias. Proposed: exclude `first_anc_month`.

## E. Other checks

1. **Very young mothers:** how many mothers were under 15? These rows are kept (they are genuine), but we note the count.
2. **Birth-weight heaping (LBW cohort):** recalled or rounded weights pile up at round numbers. The LBW cut-off is *below* 2,500 g, so a heap at exactly 2,500 g pushes some truly low weights into "normal". We count weights of exactly 2,500 g and in the 2,400 to 2,600 g band. `birth_weight_g` is used here only for this check.
3. **Imputed intervals:** later pregnancies with an unknown interval were given the cohort's median (`interval_imputed = 1`). How many are there, and is their outcome rate different?

In [ ]:
pd.DataFrame(
    {
        c: {
            "mothers under 15 (train)": int(d["maternal_age"].lt(15).sum()),
            "mothers under 18 (train)": int(d["maternal_age"].lt(18).sum()),
            "events among under 15": int(d.loc[d["maternal_age"].lt(15), OUTCOMES[c]].sum()),
        }
        for c, d in train.items()
    }
).T

In [ ]:
bw = train["lbw"]["birth_weight_g"]
heap = pd.Series(
    {
        "rows with a birth weight": int(bw.notna().sum()),
        "exactly 2,500 g": int(bw.eq(2500).sum()),
        "exactly 2,500 g (%)": 100 * bw.eq(2500).mean(),
        "2,400-2,600 g": int(bw.between(2400, 2600).sum()),
        "2,400-2,600 g (%)": 100 * bw.between(2400, 2600).mean(),
        "share of the 2,400-2,600 band at exactly 2,500 g (%)": 100 * bw.eq(2500).sum() / bw.between(2400, 2600).sum(),
        "multiple of 500 g (%)": 100 * (bw % 500 == 0).mean(),
        "multiple of 100 g (%)": 100 * (bw % 100 == 0).mean(),
        "LBW (<2,500 g) (%)": 100 * bw.lt(2500).mean(),
    }
)
heap.to_frame("LBW cohort, training rows (unweighted)")

In [ ]:
# Histogram of birth weights around the cut-off (aggregate counts per 100 g bin).
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.hist(bw[bw.between(1500, 3500)], bins=np.arange(1500, 3550, 50), color="steelblue")
ax.axvline(2500, color="black", ls="--", lw=1)
ax.set_xlabel("Birth weight (g), 50 g bins")
ax.set_ylabel("Births")
ax.set_title("LBW cohort (train): birth weights near the 2,500 g cut-off (dashed)")
plt.tight_layout()
plt.show()

In [ ]:
imp = []
for c, d in train.items():
    for lv, g in d[d["first_pregnancy"] == 0].groupby("interval_imputed"):
        rate, lo, hi = rate_ci(g, OUTCOMES[c])
        imp.append(
            {
                "cohort": c,
                "interval_imputed": lv,
                "n (later pregnancies)": len(g),
                "events": int(g[OUTCOMES[c]].sum()),
                "rate_per_1000": rate,
                "ci_low": lo,
                "ci_high": hi,
                "few_events": int(g[OUTCOMES[c]].sum()) < 30,
            }
        )
pd.DataFrame(imp).set_index(["cohort", "interval_imputed"])

In [ ]:
# Are the imputed intervals concentrated in twins? (counts only)
pd.DataFrame(
    {
        c: {
            "imputed rows": int(d["interval_imputed"].sum()),
            "of which twins": int((d["interval_imputed"].eq(1) & d["multiple"].eq(1)).sum()),
            "twin share among imputed (%)": 100 * d.loc[d["interval_imputed"] == 1, "multiple"].mean(),
            "twin share among other later pregnancies (%)": 100
            * d.loc[(d["interval_imputed"] == 0) & (d["first_pregnancy"] == 0), "multiple"].mean(),
        }
        for c, d in train.items()
    }
).T

**Result:** 

- **Mothers under 15 (training rows, kept):** perinatal 68 (5 deaths), perinatal ANC 31 (1), LBW 11 (2 LBW). Under 18: 1,302 / 630 / 314. Too few under 15 to say anything; they stay in under `maternal_age`.
- **Birth-weight heaping (LBW cohort, 7,076 training rows):** 372 weights (5.3%) are **exactly 2,500 g**, and 622 (8.8%) fall in 2,400 to 2,600 g, so 60% of that band sits on the single value 2,500. 99.4% of weights are multiples of 100 g and 44% multiples of 500 g. With LBW at 8.2% (unweighted), the 2,500 g heap is two-thirds the size of the whole outcome group. Some true LBW babies are likely recorded as 2,500 g and so counted as normal: the outcome is probably under-counted, and model results should be read with that in mind (report note, no change to the data).
- **Imputed intervals:** few rows (59 / 34 / 18 later pregnancies) but very high rates (deaths 182 and 180 per 1,000, LBW 483; 9 / 4 / 12 events). **Every imputed row is a twin** (100% vs 2 to 4% twins among other later pregnancies), so `interval_imputed` is in practice a marker for one twin of a pair whose interval was missing. This is a question for `clean_cohort()` (`dataset.py`), raised for Jeff, not changed here: twins should probably not be given the median interval. Meanwhile `interval_imputed` mostly duplicates part of `multiple`.

## F. Correlations

**Between candidates:** a Spearman matrix of the numeric and binary candidates for each cohort. Education (0 to 3), wealth quintile (1 to 5) and urban residence (1 = urban) are coded as ordered numbers for this; `interval_group` is left out because it is a grouping of `preceding_interval_m` and `first_pregnancy`. Pairs with |r| > 0.7 are listed: those carry largely the same information, and one of each pair is usually enough (especially for logistic regression).

**With the outcome:** for each candidate, the Spearman correlation with the outcome, and for binary candidates the weighted rate ratio (rate when 1 / rate when 0).

In [ ]:
CODES = {
    "education": {"No education": 0, "Primary": 1, "Secondary": 2, "Higher": 3},
    "wealth_quintile": {"Poorest": 1, "Poorer": 2, "Middle": 3, "Richer": 4, "Richest": 5},
    "residence": {"Rural": 0, "Urban": 1},
}


def numeric_candidates(c):
    d = train[c]
    cols = [v for v in candidates[c] if v != "interval_group"]
    X = pd.DataFrame(index=d.index)
    for v in cols:
        X[v] = d[v].map(CODES[v]) if v in CODES else d[v].astype(float)
    return X


corr = {}
for c in COHORTS:
    X = numeric_candidates(c)
    r = X.corr(method="spearman")
    corr[c] = r
    fig, ax = plt.subplots(figsize=(9, 7.5))
    im = ax.imshow(r, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(r)), r.columns, rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(len(r)), r.index, fontsize=8)
    for i in range(len(r)):
        for j in range(len(r)):
            ax.text(j, i, f"{r.iloc[i, j]:.2f}", ha="center", va="center", fontsize=6)
    fig.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(f"{c}: Spearman correlation between candidate features (train)")
    plt.tight_layout()
    plt.show()

In [ ]:
pairs = []
for c, r in corr.items():
    cols = list(r.columns)
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            if abs(r.iloc[i, j]) > 0.7:
                pairs.append({"cohort": c, "feature_1": cols[i], "feature_2": cols[j], "spearman": r.iloc[i, j]})
high_pairs = pd.DataFrame(pairs).sort_values(["cohort", "spearman"], key=lambda s: s if s.dtype == object else -s.abs())
high_pairs.reset_index(drop=True)

In [ ]:
uni = []
for c in COHORTS:
    d, y = train[c], OUTCOMES[c]
    X = numeric_candidates(c)
    for v in X.columns:
        ok = X[v].notna()
        rho = pd.concat([X.loc[ok, v], d.loc[ok, y]], axis=1).corr(method="spearman").iloc[0, 1]
        row = {"cohort": c, "feature": v, "spearman_with_outcome": rho}
        if set(X.loc[ok, v].unique()) <= {0.0, 1.0}:
            g1, g0 = d[ok & X[v].eq(1)], d[ok & X[v].eq(0)]
            r1, r0 = weighted_rate(g1, y), weighted_rate(g0, y)
            row.update(
                {
                    "rate_if_1": r1,
                    "rate_if_0": r0,
                    "rate_ratio": r1 / r0,
                    "events_if_1": int(g1[y].sum()),
                    "share_1_%": 100 * np.average(X.loc[ok, v], weights=d.loc[ok, "weight"]),
                }
            )
        uni.append(row)
univariate = pd.DataFrame(uni).set_index(["cohort", "feature"])
pd.options.display.float_format = "{:,.3f}".format
display(univariate)
pd.options.display.float_format = "{:,.2f}".format

The ANC content items (BP, urine, blood, iron) are recorded as 0 for women with no ANC at all, so part of their signal may simply be "had any ANC" (note `no_anc` and `blood_tested` above). The cell below repeats their rate ratios among women with at least one visit.

In [ ]:
items = ["bp_measured", "urine_tested", "blood_tested", "iron_tablets"]
rows = []
for c in ["perinatal_anc", "lbw"]:
    d, y = train[c], OUTCOMES[c]
    d = d[d["anc_visits"] >= 1]
    for v in items:
        g1, g0 = d[d[v] == 1], d[d[v] == 0]
        rows.append(
            {
                "cohort": c,
                "item (women with 1+ visits)": v,
                "rate_if_1": weighted_rate(g1, y),
                "rate_if_0": weighted_rate(g0, y),
                "rate_ratio": weighted_rate(g1, y) / weighted_rate(g0, y),
                "events_if_0": int(g0[y].sum()),
                "few_events_if_0": int(g0[y].sum()) < 30,
            }
        )
pd.DataFrame(rows).set_index(["cohort", "item (women with 1+ visits)"])

**Result:** 

- **Pairs with |Spearman| > 0.7** (all expected, by construction or biology):
  - `anc_visits` / `anc4plus`: 0.85 (perinatal ANC), 0.82 (LBW)
  - `no_anc` / `blood_tested`: -0.75 (both ANC cohorts): women with no ANC are recorded as not tested
  - `maternal_age` / `prior_live_births`: 0.74 to 0.75 (all three)
  - `prior_live_births` / `first_pregnancy`: -0.73 to -0.75 (all three)
  - `preceding_interval_m` / `short_interval`: -0.74 (perinatal), -0.72 (perinatal ANC)
- **Strongest univariate signals (weighted rate ratio, 1 vs 0):** twins 4.2 (perinatal), 5.0 (perinatal ANC), 7.7 (LBW); previous stillbirth 3.2 / 4.5 / 1.3; previous loss 1.6 / 2.0 / 1.3; no ANC 2.0 (perinatal ANC), 1.3 (LBW); `interval_imputed` 5.5 to 6.1 (few events, twins); short interval 1.2 in all three. Spearman with the outcome is small for every continuous candidate (|r| under 0.05), as expected for a rare binary outcome.
- **ANC content items:** in the perinatal ANC cohort they look protective overall (RR 0.58 to 0.74) but much of that is "had any ANC". Among women with 1+ visits: BP measured 1.03 (no signal), blood tested 0.75 (only 22 deaths among the untested), urine tested 0.79, iron 0.74. In the LBW cohort none points in a consistent direction (0.97 to 1.41).

Consequences: keep one of `anc_visits` / `anc4plus`; `blood_tested` adds nothing beyond `no_anc`; `short_interval` and `interval_group` repeat `preceding_interval_m` + `first_pregnancy`. `maternal_age` and `prior_live_births` are correlated but measure different things; keep both (tree models are not affected, and 0.75 is manageable for logistic regression).

## G. Summary: proposed `SELECTED` lists

Every list starts with the design columns: the outcome, `split`, `cluster_id`, `country`, `weight`, `interval_imputed`. `country` is kept for stratifying and reporting, not as a feature. The rest are the proposed features.

**Open questions, answered on the pooled training rows**

| Question | Answer |
|---|---|
| 1. `anc_visits` or `anc4plus`? | For perinatal death the visits gradient disappears in 9-month pregnancies (it is mostly gestation); for LBW a 4+ visit advantage survives. Use `anc4plus` + `no_anc`. **`anc_visits`: proposed exclude, Jeff to decide.** |
| 2. Wealth and residence | The "richer/urban = higher" pattern persists within facility deliveries, so the facility-reporting explanation is not supported; mostly Tanzanian. LBW shows the opposite (richest lowest), inconsistent by country. **Proposed, Jeff to decide.** |
| 3. Parity | U-shape holds pooled for perinatal death. `first_pregnancy` + `prior_live_births` misses the 5+ jump in a linear model; trees handle it, logistic regression needs a binned or 5+ term in its pipeline. |
| 4. Early booking | The 1st-trimester excess disappears among women with no previous loss: reverse causation. Exclude `first_anc_month`. |

**Include, all three cohorts**
- `maternal_age`: established risk (<18, 35+); correlated 0.75 with `prior_live_births` but a different factor.
- `prior_live_births`: right arm of the parity U (5+).
- `first_pregnancy`: left arm of the U and the reason there is no interval; not identical to parity 0.
- `prior_stillbirth`: strongest history factor for death (RR 3.2 to 4.5).
- `prior_loss`: RR 1.3 to 2.0; also adjusts for the early-booking reverse causation.
- `preceding_interval_m`: short interval RR 1.2; continuous form, binned in the logistic regression pipeline if needed.
- `multiple`: strongest single factor (RR 4.2 to 7.7).

**Include, ANC cohorts**
- `no_anc`: RR 2.0 for death, 1.3 for LBW; the one ANC signal that survives in 9-month pregnancies (on few events).
- `anc4plus`: the WHO-style threshold; carries the surviving LBW signal; 0.82 to 0.85 with `anc_visits`, so only one of them.
- `urine_tested`, `iron_tablets` (perinatal ANC only): weak (RR about 0.75 to 0.8 among attenders); low-cost to test in the model.

**Include, LBW only**
- `education`: higher education lowest LBW (58 vs about 84); flat for perinatal death, so excluded from the perinatal cohorts.

**Proposed, Jeff to decide**
- `wealth_quintile`, `residence`: signal is real in the data but runs against clinical expectation for perinatal death and is not explained by delivery place (evidence in section B). Options: include and report it as a likely reporting/referral artefact, or exclude to keep the model clinically interpretable.
- `anc_visits`: proposed **exclude** in favour of `anc4plus` + `no_anc` (gestation confounding, section A; 0.85 correlation).

**Exclude**
- `interval_group`, `short_interval`: derived from `preceding_interval_m` and `first_pregnancy` (|r| 0.72 to 0.74).
- `first_anc_month`: reverse causation and survival bias (section D).
- `bp_measured`: no signal among attenders (RR 1.03 perinatal ANC).
- `blood_tested`: duplicates `no_anc` (r -0.75); little signal beyond it.
- `urine_tested`, `iron_tablets` in LBW: no consistent direction.
- `education` in the perinatal cohorts: flat (Spearman about 0).
- Never-model columns (section 2) are excluded throughout.

**For Jeff (not changed here):** every `interval_imputed` row is a twin (section E); `clean_cohort()` may want to treat twins' missing interval differently.

In [ ]:
DESIGN = ["split", "cluster_id", "country", "weight", "interval_imputed"]
CORE = [
    "maternal_age",
    "prior_live_births",
    "first_pregnancy",
    "prior_stillbirth",
    "prior_loss",
    "preceding_interval_m",
    "multiple",
]

PROPOSED_SELECTED = {
    "perinatal": ["perinatal_death"] + DESIGN + CORE,
    "perinatal_anc": ["perinatal_death"] + DESIGN + CORE
    + ["no_anc", "anc4plus", "urine_tested", "iron_tablets"],
    "lbw": ["low_birth_weight"] + DESIGN + CORE + ["education", "no_anc", "anc4plus"],
}

# Proposed, Jeff to decide (evidence in sections A and B).
JEFF_TO_DECIDE = {
    "perinatal": ["wealth_quintile", "residence"],
    "perinatal_anc": ["wealth_quintile", "residence", "anc_visits (proposed exclude)"],
    "lbw": ["wealth_quintile", "residence", "anc_visits (proposed exclude)"],
}

for c in COHORTS:
    cols = PROPOSED_SELECTED[c]
    assert all(col in train[c].columns for col in cols), c
    print(f"{c} ({len(cols)} columns): {cols}")
    print(f"    Jeff to decide: {JEFF_TO_DECIDE[c]}")